# Batch BLS Pipeline: 500 Random TESS Targets

**Purpose**: Randomly select 500 bright stars from the TIC catalog, download their TESS light curves, run the BLS transit search pipeline (same as Person1's), and output a CSV with columns **identical to `master_candidates.csv`**.

**Workflow**:
```
1. Query TIC  Select 500 random stars with Tmag < 12 (bright enough for good TESS data)
2. For each star:
   a. Search MAST for SPOC light curves, pick the sector with the most data
   b. Download + flatten + run BLS + bootstrap errors + vet (False Positive flag)
   c. Append 1-row summary to the batch table
3. Save -> master_candidates_500.csv  (same 8 columns as Person1's original)
```

**Output**: `master_candidates_500.csv` with columns: `Target, Sector, Period (d), Period Err (d), Depth (ppm), Duration (hr), SNR, False Positive`.

## 0. Install Dependencies

In [1]:
%pip install -q numpy pandas astropy astroquery lightkurve

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [ ]:
import os, sys, time as timer, warnings
import numpy as np
import pandas as pd
from astropy.timeseries import BoxLeastSquares
import lightkurve as lk

warnings.filterwarnings('ignore')

# ---------------- Parameters ----------------
N_TARGETS          = 200          # number of random targets to process
TIC_TMAG_MAX        = 12.0         # only stars brighter than this (ensures decent SNR)
RANDOM_SEED         = 13           # reproducibility
N_BOOTSTRAP         = 50           # bootstrap draws per target (50 = fast screening; 200 = final)
MIN_PERIOD          = 0.5          # days
MAX_PERIOD          = 40.0         # days (v2 widened from 15)
SNR_THRESHOLD       = 7.0          # depth_snr below this -> False Positive
PER_TARGET_TIMEOUT  = 180          # seconds hard timeout per target
BATCH_SIZE          = 50           # save intermediate CSV every N targets
TESS_CACHE_DIR      = 'tess_cache'
OUTPUT_DIR          = 'outputs'
os.makedirs(TESS_CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'output.csv')
CHECKPOINT_CSV = os.path.join(OUTPUT_DIR, 'checkpoint.csv')  # intermediate save

print(f'Will process {N_TARGETS} targets, Tmag < {TIC_TMAG_MAX}')
print(f'BLS period range: {MIN_PERIOD} - {MAX_PERIOD} days')
print(f'Timeout per target: {PER_TARGET_TIMEOUT}s, batch size: {BATCH_SIZE}')

Will process 500 targets, Tmag < 12.0
BLS period range: 0.5 - 40.0 days
Timeout per target: 180s, batch size: 50


C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


## 2. Get 500 Random Targets from TIC

In [3]:
from astroquery.mast import Catalogs

def _f(x):
    '''Safely convert masked table element to float.'''
    try:
        if x is None or np.ma.is_masked(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

print(f'Querying TIC for stars with Tmag < {TIC_TMAG_MAX} ...')

# Query a large pool; we will randomly sample from it
# Use a cone search at a random position to get a diverse field
import random
random.seed(RANDOM_SEED)

all_targets = []
# Query several random sky positions to get a diverse sample
for _ in range(20):
    ra = random.uniform(0, 360)
    dec = random.uniform(-80, 80)  # avoid poles (fewer TESS sectors)
    try:
        cat = Catalogs.query_region(f'{ra} {dec}', catalog='TIC', radius=2.0)
        if cat is not None and len(cat) > 0:
            # Filter: bright stars with valid parameters
            tmags = [_f(r['Tmag']) for r in cat]
            valid = [i for i, t in enumerate(tmags) if np.isfinite(t) and t <= TIC_TMAG_MAX]
            for i in valid:
                all_targets.append({
                    'target': str(cat[i]['ID']),
                    'tic_id': int(cat[i]['ID']),
                    'tmag': tmags[i]
                })
    except Exception:
        continue
    if len(all_targets) >= N_TARGETS * 3:
        break

print(f'Retrieved pool: {len(all_targets)} bright stars')

# Randomly sample N_TARGETS
random.shuffle(all_targets)
target_list = all_targets[:N_TARGETS]
print(f'Sampled {len(target_list)} random targets')
print(f'Tmag range: {min(t["tmag"] for t in target_list):.1f} - {max(t["tmag"] for t in target_list):.1f}')

Querying TIC for stars with Tmag < 12.0 ...
Retrieved pool: 1588 bright stars
Sampled 500 random targets
Tmag range: 5.8 - 12.0


## 3. Core Pipeline Functions

Same logic as Person1's `exoplanet_bls_pipeline.py`, adapted for notebook use.

In [4]:
def extract_arrays(lc_flat):
    '''Extract time, flux, flux_err from a flattened light curve.'''
    time = np.asarray(lc_flat.time.value, dtype=float)
    flux = np.asarray(lc_flat.flux.value, dtype=float)
    if getattr(lc_flat, 'flux_err', None) is not None:
        flux_err = np.asarray(lc_flat.flux_err.value, dtype=float)
    else:
        flux_err = None
    good = np.isfinite(time) & np.isfinite(flux)
    if flux_err is not None:
        good &= np.isfinite(flux_err)
        flux_err = flux_err[good]
    else:
        mad = np.nanmedian(np.abs(flux - np.nanmedian(flux)))
        flux_err = np.full(good.sum(), 1.4826 * mad)
    time = time[good]
    flux = flux[good]
    order = np.argsort(time)
    return time[order], flux[order], flux_err[order]


def run_bls(time, flux, flux_err, min_period=MIN_PERIOD, max_period=MAX_PERIOD,
            n_periods=15000, n_durations=10):
    '''Run BoxLeastSquares search; return (bls, result, best).'''
    bls = BoxLeastSquares(time, flux, dy=flux_err)
    periods = np.linspace(min_period, max_period, n_periods)
    durations = np.linspace(0.5/24.0, 6.0/24.0, n_durations)
    result = bls.power(periods, durations, objective='snr')
    idx = int(np.argmax(result.power))
    best = {
        'period':   float(result.period[idx]),
        'duration': float(result.duration[idx]),
        't0':       float(result.transit_time[idx]),
        'power':    float(result.power[idx]),
        'depth':    float(result.depth[idx]),
        'depth_err': float(result.depth_err[idx]),
        'depth_snr': float(result.depth_snr[idx]),
    }
    return bls, result, best


def bootstrap_errors(time, flux, flux_err, best, n_boot=N_BOOTSTRAP, rng_seed=42):
    '''Block bootstrap to estimate period / depth uncertainties.'''
    rng = np.random.default_rng(rng_seed)
    n = len(time)
    block_size = max(50, n // 50)
    n_blocks = int(np.ceil(n / block_size))
    lo = best['period'] * 0.95
    hi = best['period'] * 1.05
    boot_periods_grid = np.linspace(lo, hi, 300)
    durations = np.array([best['duration']])
    boot_periods, boot_depths = [], []
    for _ in range(n_boot):
        starts = rng.integers(0, n_blocks, size=n_blocks) * block_size
        idx = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])
        idx = idx[idx < n]
        if len(idx) < 10:
            continue
        t_b, f_b, e_b = time[idx], flux[idx], flux_err[idx]
        order = np.argsort(t_b)
        t_b, f_b, e_b = t_b[order], f_b[order], e_b[order]
        try:
            bls_b = BoxLeastSquares(t_b, f_b, dy=e_b)
            res_b = bls_b.power(boot_periods_grid, durations, objective='snr')
            j = int(np.argmax(res_b.power))
            boot_periods.append(res_b.period[j])
            boot_depths.append(res_b.depth[j])
        except Exception:
            continue
    boot_periods = np.array(boot_periods)
    boot_depths  = np.array(boot_depths)
    return {
        'period_err': float(np.std(boot_periods)) if len(boot_periods) > 5 else np.nan,
        'depth_err':  float(np.std(boot_depths))  if len(boot_depths)  > 5 else np.nan,
    }


def vet_candidate(bls, best, snr_threshold=SNR_THRESHOLD):
    '''Apply false-positive rules to the candidate.'''
    flags = []
    if best['depth'] * 1e6 > 30000:
        flags.append('too_deep')
    if best['duration'] / best['period'] > 0.15:
        flags.append('too_long')
    if best['depth_snr'] < snr_threshold:
        flags.append('low_snr')
    return len(flags) > 0


def process_one_target(target_name, tic_id=None, verbose=True):
    '''Download light curve, run BLS, return a 1-row summary DataFrame.'''
    display_name = target_name if tic_id is None else f'TIC {tic_id}'
    if verbose:
        print(f'  Searching MAST for {display_name} ...')
    search = lk.search_lightcurve(f'TIC {tic_id}', author='SPOC') if tic_id else lk.search_lightcurve(target_name, author='SPOC')
    if search is None or len(search) == 0:
        # fallback: search without author filter
        search = lk.search_lightcurve(f'TIC {tic_id}') if tic_id else lk.search_lightcurve(target_name)
    if search is None or len(search) == 0:
        return None
    # Pick sector with most exposures
    try:
        lc = search[0].download(download_dir=TESS_CACHE_DIR)
    except Exception:
        return None
    if lc is None:
        return None
    sector = int(getattr(lc, 'sector', -1))
    target_label = target_name if target_name else f'TIC {tic_id}'
    if verbose:
        print(f'  Sector {sector}, flattening ...')
    lc_flat = lc.remove_nans().flatten(window_length=401)
    time, flux, flux_err = extract_arrays(lc_flat)
    if len(time) < 200:
        return None
    if verbose:
        print(f'  Running BLS ({len(time)} cadences) ...')
    bls, result, best = run_bls(time, flux, flux_err)
    if verbose:
        print(f'  P={best["period"]:.4f}d  depth={best["depth"]*1e6:.0f}ppm  SNR={best["depth_snr"]:.1f}')
    boot = bootstrap_errors(time, flux, flux_err, best)
    is_fp = vet_candidate(bls, best)
    summary = pd.DataFrame([{
        'Target':        target_label,
        'Sector':        sector,
        'Period (d)':    round(best['period'], 5),
        'Period Err (d)':round(boot['period_err'], 5) if np.isfinite(boot['period_err']) else np.nan,
        'Depth (ppm)':   round(best['depth'] * 1e6, 1),
        'Duration (hr)': round(best['duration'] * 24, 3),
        'SNR':           round(best['depth_snr'], 2),
        'False Positive':is_fp,
    }])
    return summary

## 4. Batch Runner

Process all 500 targets with timeout protection, intermediate saves, and failure logging.

In [5]:
import concurrent.futures as cf

all_tables = []
failed = []
t_start = timer.time()
completed_count = 0

# Check for existing checkpoint (resume support)
if os.path.exists(CHECKPOINT_CSV):
    existing = pd.read_csv(CHECKPOINT_CSV)
    all_tables = [existing]
    done_names = set(existing['Target'].tolist())
    print(f'Resumed from checkpoint: {len(existing)} targets already processed')
else:
    done_names = set()

for i, t in enumerate(target_list, 1):
    display_name = f'TIC {t["tic_id"]}'
    if display_name in done_names:
        continue
    print(f'\n[{i}/{N_TARGETS}] {display_name}  (Tmag={t["tmag"]:.1f})')
    t0 = timer.time()
    with cf.ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(process_one_target, None, t['tic_id'], verbose=True)
        try:
            table = future.result(timeout=PER_TARGET_TIMEOUT)
            if table is not None:
                all_tables.append(table)
                completed_count += 1
            else:
                failed.append(display_name)
                print(f'  -> FAILED (no data / download error)')
        except cf.TimeoutError:
            failed.append(display_name)
            print(f'  -> TIMEOUT after {PER_TARGET_TIMEOUT}s')
        except Exception as e:
            failed.append(display_name)
            print(f'  -> FAILED: {type(e).__name__}: {str(e)[:80]}')
    print(f'  took {timer.time()-t0:.1f}s | completed: {completed_count} | failed: {len(failed)}')
    # Intermediate save every BATCH_SIZE
    if completed_count > 0 and completed_count % BATCH_SIZE == 0:
        checkpoint = pd.concat(all_tables, ignore_index=True)
        checkpoint.to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')
        print(f'  [checkpoint saved: {len(checkpoint)} rows]')

print(f'\n{"="*60}')
print(f'BATCH COMPLETE in {timer.time()-t_start:.0f}s')
print(f'Succeeded: {completed_count}  |  Failed: {len(failed)}')
print(f'{"="*60}')


[1/500] TIC 293694800  (Tmag=10.7)
  Searching MAST for TIC 293694800 ...
  Sector 61, flattening ...
  Running BLS (9257 cadences) ...
  P=3.3179d  depth=380ppm  SNR=7.0
  took 38.1s | completed: 1 | failed: 0

[2/500] TIC 156919710  (Tmag=11.6)
  Searching MAST for TIC 156919710 ...
  Sector 6, flattening ...
  Running BLS (986 cadences) ...
  P=6.5228d  depth=1231ppm  SNR=6.8
  took 19.4s | completed: 2 | failed: 0

[3/500] TIC 232050760  (Tmag=11.4)
  Searching MAST for TIC 232050760 ...
  Sector 7, flattening ...
  Running BLS (1067 cadences) ...
  P=18.9398d  depth=1575ppm  SNR=6.2
  took 26.6s | completed: 3 | failed: 0

[4/500] TIC 157077690  (Tmag=11.9)
  Searching MAST for TIC 157077690 ...
  Sector 6, flattening ...
  Running BLS (986 cadences) ...
  P=2.1723d  depth=1119ppm  SNR=15.6
  took 22.0s | completed: 4 | failed: 0

[5/500] TIC 232041402  (Tmag=11.3)
  Searching MAST for TIC 232041402 ...
  Sector 6, flattening ...
  Running BLS (972 cadences) ...
  P=17.0437d  dep

## 5. Save Final CSV and Summary

In [6]:
if all_tables:
    final = pd.concat(all_tables, ignore_index=True)
    # Ensure column order matches master_candidates.csv exactly
    COL_ORDER = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
                 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']
    final = final[COL_ORDER]
    final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    # Also save final checkpoint
    final.to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')

    print(f'Final table saved -> {OUTPUT_CSV}')
    print(f'Rows: {len(final)}  |  Columns: {list(final.columns)}')
    print(f'\nColumn comparison with master_candidates.csv:')
    print(f'  Output: {list(final.columns)}')
    ref = pd.read_csv('attachments/master_candidates.csv', nrows=0)
    print(f'  Master: {list(ref.columns)}')
    assert list(final.columns) == list(ref.columns), 'Column mismatch!'
    print('  -> Columns match !')
    print(f'\nSummary statistics:')
    print(f'  SNR > 10, not FP : {len(final[(final["SNR"]>10) & (~final["False Positive"])])}')
    print(f'  SNR 7-10, not FP  : {len(final[(final["SNR"]>=7) & (final["SNR"]<=10) & (~final["False Positive"])])}')
    print(f'  Flagged FP        : {len(final[final["False Positive"]])}')
    print(f'\nFirst 10 rows:')
    print(final.head(10).to_string(index=False))
else:
    print('No targets succeeded. Check network / MAST access.')

if failed:
    print(f'\nFailed targets ({len(failed)}):')
    for f in failed[:20]:
        print(f'  {f}')
    if len(failed) > 20:
        print(f'  ... and {len(failed)-20} more')

Final table saved -> outputs\master_candidates_500.csv
Rows: 500  |  Columns: ['Target', 'Sector', 'Period (d)', 'Period Err (d)', 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

Column comparison with master_candidates.csv:
  Output: ['Target', 'Sector', 'Period (d)', 'Period Err (d)', 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']
  Master: ['Target', 'Sector', 'Period (d)', 'Period Err (d)', 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']
  -> Columns match !

Summary statistics:
  SNR > 10, not FP : 139
  SNR 7-10, not FP  : 72
  Flagged FP        : 289

First 10 rows:
       Target  Sector  Period (d)  Period Err (d)  Depth (ppm)  Duration (hr)    SNR  False Positive
TIC 293694800      61     3.31785         0.06627        380.4           5.40   6.98            True
TIC 156919710       6     6.52283         0.14106       1230.7           2.95   6.76            True
TIC 232050760       7    18.93983         0.46561       1575.2           1.10   6.18    

## 6. Notes

- **Column format**: The output CSV has exactly the same 8 columns as `master_candidates.csv`, produced by the same pipeline logic (BLS + bootstrap + vet).
- **Random seed**: Set to 42 for reproducibility; change `RANDOM_SEED` to get a different sample.
- **Resume support**: If the run is interrupted, re-running the notebook will resume from `checkpoint_500.csv`.
- **Timeout**: Each target has a 180-second hard timeout; targets that time out or fail to download are skipped and logged.
- **Performance**: With 500 targets and `n_bootstrap=50`, expect ~30-90 minutes runtime depending on network speed.
- **This CSV can be directly fed into the three-step quality control notebooks** (odd-even check, radius check, NASA cross-match).